# Lab 11 — End-to-End ML Workflow on Arizona Bridges
**CE/ARCE 2XX · AI Literacy for Civil and Architectural Engineering** · Week 11 lab · about 75 minutes

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TensorBuildLab/AI-Literacy-CE-ARCE/blob/main/labs/lab11-bridges-end-to-end/Lab11_Bridges_End_to_End.ipynb)

So far you have practiced one piece of machine learning at a time. Today you do the **whole job**, start to finish, on real public data: every bridge in Arizona from the 2024 National Bridge Inventory. The question is one a state bridge engineer really asks: **which bridge decks are likely in poor condition, so we can send inspectors and repair money there first?**

This notebook is also your **final-project template**. Sections marked **🧩 PROJECT TEMPLATE** are the steps every project needs. Your team can copy this notebook and swap in your own data and question.

### What you will be able to do
1. Frame an engineering question as a machine-learning task: users, decision, target, and success metric.
2. Explore and clean a real infrastructure dataset (maps, plots, missing values, special codes).
3. Build features, including one-hot encoding of categories, and split data fairly with `stratify`.
4. Compare a baseline, logistic regression, and a random forest, and explain why accuracy misleads when the class you care about is rare.
5. Read a confusion matrix, choose a decision threshold, and interpret feature importance with care.
6. Write a short engineering memo that states results *and* caveats.

### Before you start
1. Click **Open in Colab** above (or open this file from the course GitHub page).
2. In Colab, choose **File → Save a copy in Drive**. Work in *your copy* so nothing is lost.
3. Run cells from top to bottom with **Shift + Enter**. Cells marked `# TODO` are yours to complete.

**Runtime:** the default CPU runtime is fine.
**Data:** `arizona_bridges_nbi2024.csv` — FHWA National Bridge Inventory 2024, Arizona (U.S. Government work, public domain). See the course `data/` README for column meanings.

**AI-use label:** **AI-Assisted** — you may ask an AI tool for explanations or debugging help. Fill in the AI-use acknowledgment at the end. You are responsible for every line.

### How to use this notebook as a project template
The workflow has eight steps. The ones marked **🧩 PROJECT TEMPLATE** are the skeleton of every final project:

| Step | What you do | In your project, change… |
|---|---|---|
| 1. Frame | Who uses the result, what decision, what target, what metric | the problem card |
| 2. Explore | Look at the data before modeling | file name, plots |
| 3. Features | Choose inputs (X) and target (y); clean and encode | the column lists |
| 4. Split + baseline | Hold out test data; beat a "dumb" rule first | the baseline rule |
| 5. Models | Try a simple and a flexible model | model choices |
| 6. Evaluate | Metrics that match the decision; confusion matrix | the metric |
| 7. Explain | What drives the predictions? | nothing (reuse) |
| 8. Communicate | A short memo with numbers *and* caveats | the memo |

*Regression project (predicting a number)?* Swap the classifiers for `LinearRegression` / `RandomForestRegressor`,
the baseline for "always predict the training mean", and the metrics for MAE and R² (as in Labs 4–5).

In [ ]:
# Setup: run this cell first (Shift + Enter). It loads libraries and a helper to fetch course data.
import os, urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, recall_score, precision_score, confusion_matrix, ConfusionMatrixDisplay
BASE_URL = "https://raw.githubusercontent.com/TensorBuildLab/AI-Literacy-CE-ARCE/main/data/"

def get_data(name):
    """Download a course data file once and return its local path, e.g. get_data("concrete_strength.csv")."""
    if not os.path.exists(name):
        folder = os.path.dirname(name)
        if folder:
            os.makedirs(folder, exist_ok=True)
        urllib.request.urlretrieve(BASE_URL + name, name)
    return name

print("Setup complete.")

## Part 1 — Frame the question (5 min) · 🧩 PROJECT TEMPLATE · Step 1
Before any code, write down **who** will use the model and **what decision** it supports. This "problem card"
keeps the whole project honest.

**Background: NBI condition ratings.** Every bridge on a public road in the U.S. is inspected, usually every
2 years. Inspectors rate the **deck** (the riding surface), the **superstructure** (beams/girders), and the
**substructure** (piers, abutments) from **9 (excellent) to 0 (failed)**. FHWA groups ratings as
**Good (7–9)**, **Fair (5–6)**, and **Poor (4 or less)**. Culverts get a separate culvert rating instead.

| Problem card | Our answer |
|---|---|
| **Users** | Arizona bridge program engineers who schedule inspections and repairs |
| **Decision** | Which bridges get a closer look (or repair funds) first |
| **Unit of analysis** | One bridge (one row) |
| **Target (y)** | `poor` = 1 if the deck rating is 4 or less, else 0 |
| **Inputs (X)** | Things known *without* a new inspection: age, traffic, size, material, design |
| **Success metric** | **Recall** on poor decks (don't miss them), with a manageable number of false alarms |
| **Baseline to beat** | "Every deck is fine" (always predict *not poor*) |
| **Not in scope** | Replacing inspections. A model can only help *prioritize* them. |

**✍️ Question:** Why should the inputs be things we know *without* a new inspection? What would go wrong if we used `superstructure_condition` as an input?

*Your answer:* 

### 🤔 Why predict a rating we already have?
Fair question: every bridge in this file already has a 2024 deck rating. Here is the honest answer.
- **Risk factors.** The model shows *which* attributes go with poor decks (age, material, traffic…). That helps
  plan preventive work before a deck gets that bad.
- **Screening where ratings are missing or old.** Some structures are not in the NBI (many small local bridges),
  and between inspections a rating can be up to 2 years out of date. A model that uses only inventory data can
  help decide where to look first.
- **A teaching stand-in.** Today the point is the *workflow*. A real early-warning tool would **forecast**: use
  features from an **earlier inventory year** (say 2019) to predict which decks *became* poor by 2024. That is a
  great final project — see idea P5 on the [project ideas page](../../project/project-ideas.md).

## Part 2 — Explore the data (15 min) · 🧩 PROJECT TEMPLATE · Step 2
Load the file and look at it before doing anything else.

In [ ]:
df = pd.read_csv(get_data("arizona_bridges_nbi2024.csv"))
print(df.shape[0], "structures,", df.shape[1], "columns")
df.head()

In [ ]:
# How many structures have NO deck rating? (blank = NaN = "not a number")
print("Missing deck rating:", df["deck_condition"].isna().sum())
print(df[df["deck_condition"].isna()]["design_type"].value_counts())

More than half the structures are **culverts** (large pipes or boxes under the road). They have no deck
rating, so they are outside our question. We keep only structures that have a deck rating.

> 💡 **Hint:** `df["deck_condition"].notna()` is True where a rating exists. Put it inside `df[ ... ]` to filter rows, and add `.copy()` at the end so later changes don't touch `df`.

In [ ]:
# TODO: keep only rows where deck_condition is NOT missing. Call the result `bridges`.
bridges = None
print(len(bridges), "bridges with a deck rating")

In [ ]:
# ✅ Self-check: run this cell to test your work above
assert bridges is not None and len(bridges) == 3762, 'Expected 3762 bridges.'
print('Correct!')

Now make the **target** column: 1 for a poor deck (rating ≤ 4), 0 otherwise.

> 💡 **Hint:** `(bridges["deck_condition"] <= 4)` gives True/False; `.astype(int)` turns True/False into 1/0.

In [ ]:
# TODO: create a column "poor" that is 1 when deck_condition <= 4 and 0 otherwise.
bridges["poor"] = None
print(bridges["poor"].value_counts())

In [ ]:
# ✅ Self-check: run this cell to test your work above
assert int(bridges['poor'].sum()) == 41, 'Expected 41 poor decks.'
print('Correct!')

In [ ]:
share = bridges["poor"].mean()
print(f"Poor decks: {bridges['poor'].sum()} of {len(bridges)} = {share:.1%}")

counts = bridges["deck_condition"].value_counts().sort_index()
colors = ["#AB0520" if r <= 4 else "#378DBD" for r in counts.index]
plt.figure(figsize=(6, 3.5))
plt.bar(counts.index.astype(int), counts.values, color=colors)
plt.xlabel("Deck condition rating (0 = failed, 9 = excellent)")
plt.ylabel("Number of bridges")
plt.title("Arizona bridge decks, 2024 (red = poor, rating ≤ 4)")
plt.show()

Only about **1 bridge in 90** has a poor deck. That is good news for drivers — and a big challenge for
machine learning. Keep this number in mind.

### A map of the bridges
Latitude and longitude are just y and x coordinates, so a scatter plot is a quick map.

In [ ]:
ok = bridges[bridges["poor"] == 0]
bad = bridges[bridges["poor"] == 1]

plt.figure(figsize=(6, 6))
plt.scatter(ok["longitude"], ok["latitude"], s=4, color="#B0B7BC", label="not poor")
plt.scatter(bad["longitude"], bad["latitude"], s=40, color="#AB0520", marker="^", label="poor deck (≤ 4)")
plt.xlabel("Longitude (degrees)")
plt.ylabel("Latitude (degrees)")
plt.title("Arizona bridges with a deck rating, NBI 2024")
plt.legend()
plt.gca().set_aspect("equal")
plt.show()

**✍️ Question:** Describe the map in 2 sentences. Where are most bridges? Do poor decks cluster anywhere?

*Your answer:* 

### Age vs. condition
The data has `year_built`, not age. Age is easy to compute: the data is from 2024.

> 💡 **Hint:** Subtract a column from a number: `2024 - bridges["year_built"]`.

In [ ]:
# TODO: add a column "age_years" = 2024 minus year_built.
bridges["age_years"] = None
bridges["age_years"].describe()

In [ ]:
# ✅ Self-check: run this cell to test your work above
assert bridges['age_years'].max() == 124 and bridges['age_years'].min() == 1
print('Correct!')

In [ ]:
ratings = sorted(bridges["deck_condition"].unique())
groups = [bridges.loc[bridges["deck_condition"] == r, "age_years"] for r in ratings]

plt.figure(figsize=(7, 4))
plt.boxplot(groups)
plt.xticks(range(1, len(ratings) + 1), [int(r) for r in ratings])
plt.xlabel("Deck condition rating")
plt.ylabel("Bridge age (years)")
plt.title("Older bridges tend to have lower deck ratings")
plt.show()
print(bridges.groupby("deck_condition")["age_years"].agg(["count", "median"]))

Each box shows the middle half of ages for one rating; the line inside is the median. Note the small
counts for ratings 3, 4, and 9 — boxes built from 2–40 bridges are shaky.

### A data-quality surprise
Real datasets hide special codes. Look at the largest "year reconstructed":

In [ ]:
print("Largest year_reconstructed:", bridges["year_reconstructed"].max())
print((bridges["year_reconstructed"] > 2024).sum(), "bridges have a reconstruction year after 2024")

A reconstruction in the year **9999**? That is a placeholder code meaning *unknown*, not a real year.
If we computed "years since reconstruction" with it we would get −7975 years. **Always check min and max values
before modeling.** We will treat 9999 as missing.

## Part 3 — Build the features (10 min) · 🧩 PROJECT TEMPLATE · Step 3
**Numeric features** we will use:
- `age_years` — years since built
- `years_since_work` — years since built *or* last reconstructed, whichever is later
- `avg_daily_traffic` (vehicles/day), `pct_truck_traffic` (%), `lanes_on`
- `main_spans`, `max_span_m`, `length_m`, `deck_width_m` (m)

**Categorical features**: `material` and `design_type`. Models need numbers, so we use **one-hot encoding**:
turn one text column into several 0/1 columns, one per category (1 = "this bridge is in this category").

In [ ]:
recon = bridges["year_reconstructed"].where(bridges["year_reconstructed"] <= 2024)   # 9999 -> NaN
last_work = recon.fillna(bridges["year_built"])        # never reconstructed -> use year built
bridges["years_since_work"] = 2024 - last_work

# 70 bridges have no truck percentage; fill with the median so we can keep them (a simple, common choice)
print("Missing truck %:", bridges["pct_truck_traffic"].isna().sum())
bridges["pct_truck_traffic"] = bridges["pct_truck_traffic"].fillna(bridges["pct_truck_traffic"].median())
bridges[["age_years", "years_since_work"]].describe().round(1)

Here is what one-hot encoding does on three example rows:

In [ ]:
example = pd.DataFrame({"material": ["Steel", "Concrete", "Steel"], "length_m": [30.0, 12.5, 48.0]})
pd.get_dummies(example, columns=["material"], dtype=int)

In [ ]:
numeric = ["age_years", "years_since_work", "avg_daily_traffic", "pct_truck_traffic", "lanes_on",
           "main_spans", "max_span_m", "length_m", "deck_width_m"]
categorical = ["material", "design_type"]
print(bridges[categorical].nunique())

> 💡 **Hint:** Select the columns with `bridges[numeric + categorical]` (adding two lists joins them), then `pd.get_dummies(..., columns=categorical, dtype=int)` like the example above.

In [ ]:
# TODO: build X with pd.get_dummies on the numeric + categorical columns, one-hot encoding the categorical ones.
X = None
y = bridges["poor"]
print(X.shape)

In [ ]:
# ✅ Self-check: run this cell to test your work above
assert X is not None and X.shape == (3762, 32), f'Expected (3762, 32); got {None if X is None else X.shape}'
print('Correct!')

## Part 4 — Split, then beat a baseline (10 min) · 🧩 PROJECT TEMPLATE · Step 4
We hold out 30% of bridges as a **test set** the models never see during training.

A **stratified split** (`stratify=y`) keeps the **same share of poor decks** in the training and test parts.
Without it, a random split could put almost all 41 poor decks in one part by bad luck.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
print(f"Train: {len(y_train)} bridges, {y_train.sum()} poor")
print(f"Test:  {len(y_test)} bridges, {y_test.sum()} poor")

Only **12 poor decks** are in the test set. Remember that when you read the results: one more or one
fewer correct poor bridge changes recall by 8 percentage points.

A **baseline** is the simplest reasonable prediction rule. Any model must beat it to be worth using. Ours is the
laziest possible rule: predict "not poor" (0) for every bridge.

> 💡 **Hint:** `accuracy_score(y_test, baseline_pred)`.

In [ ]:
# TODO: baseline predictions (all zeros) and their accuracy on the test set.
baseline_pred = np.zeros(len(y_test), dtype=int)
baseline_acc = None   # use accuracy_score(true labels, predictions)
print(f"Baseline accuracy: {baseline_acc:.1%}")

In [ ]:
# ✅ Self-check: run this cell to test your work above
assert baseline_acc is not None and abs(baseline_acc - 0.9894) < 0.001
print('Correct!')

**98.9% accuracy — while missing every single poor deck.** Accuracy is the share of *all* bridges
classified correctly; when 99% of bridges are fine, "everything is fine" scores 99%.

We need metrics that focus on the class we care about:
- **Recall (poor class)** = of the truly poor decks, what fraction did we flag? *(Did we miss any?)*
- **Precision (poor class)** = of the bridges we flagged, what fraction are truly poor? *(How many false alarms?)*

The helper below prints all of them, plus how many bridges a model would send to inspectors.

In [ ]:
results = []

def report(name, y_true, y_pred):
    """Print and store accuracy, recall, and precision for the poor class."""
    row = {
        "model": name,
        "accuracy": accuracy_score(y_true, y_pred),
        "recall_poor": recall_score(y_true, y_pred, zero_division=0),
        "precision_poor": precision_score(y_true, y_pred, zero_division=0),
        "flagged": int(y_pred.sum()),
        "poor_caught": int(((y_pred == 1) & (y_true == 1)).sum()),
    }
    results.append(row)
    print(f"{name:31s} accuracy {row['accuracy']:.1%} | recall {row['recall_poor']:.0%} | "
          f"precision {row['precision_poor']:.0%} | flagged {row['flagged']} | caught {row['poor_caught']} of {int(y_true.sum())}")

report("Baseline (all not poor)", y_test.values, baseline_pred)

## Part 5 — Try real models (10 min) · 🧩 PROJECT TEMPLATE · Step 5
**Logistic regression** (Week 4) draws a boundary using a weighted sum of the features. We scale the features
first (they range from 1 lane to 246,000 vehicles/day). `make_pipeline` chains the two steps so they always run
together.

In [ ]:
logreg = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
logreg.fit(X_train, y_train)
report("Logistic regression", y_test.values, logreg.predict(X_test))

Look closely: it is **identical to the baseline**. With so few poor decks, the cheapest way for the model
to reduce its training error is to predict "not poor" every time. *The model learned the baseline.*

**The fix: class weights.** `class_weight="balanced"` tells the model that each poor deck counts as much as
about 90 good ones (the ratio of the two classes), so missing one is expensive. This matches our engineering
judgment: a missed poor deck is far worse than an extra inspection.

> 💡 **Hint:** Copy the `logreg` cell, add `class_weight="balanced"` inside `LogisticRegression(...)`, and don't forget `.fit(X_train, y_train)`.

In [ ]:
# TODO: same pipeline, but LogisticRegression(max_iter=2000, class_weight="balanced"). Fit it and report it.
logreg_bal = None

report("Logistic regression (balanced)", y_test.values, logreg_bal.predict(X_test))

A **random forest** is a vote among many decision trees, each trained on a random sample of bridges and
features. It can capture "if old *and* steel *and* narrow…" patterns that a single weighted sum cannot. Trees
don't need scaling. `min_samples_leaf=5` stops trees from memorizing single bridges.

> 💡 **Hint:** `RandomForestClassifier(n_estimators=300, min_samples_leaf=5, class_weight="balanced", random_state=42)`, then `.fit(X_train, y_train)`.

In [ ]:
# TODO: a RandomForestClassifier with n_estimators=300, min_samples_leaf=5, class_weight="balanced",
#       random_state=42. Fit it on the training data and report it.
forest = None

report("Random forest (balanced)", y_test.values, forest.predict(X_test))

In [ ]:
# ✅ Self-check: run this cell to test your work above
assert forest is not None and logreg_bal is not None
assert recall_score(y_test, forest.predict(X_test)) > 0.3, 'Forest recall looks too low — check class_weight.'
print('Both models trained.')

In [ ]:
table = pd.DataFrame(results).set_index("model")
display(table.style.format({"accuracy": "{:.1%}", "recall_poor": "{:.0%}", "precision_poor": "{:.0%}"}))

ax = table[["accuracy", "recall_poor"]].plot.bar(figsize=(7, 4), color=["#B0B7BC", "#AB0520"], rot=15)
ax.set_ylabel("Score (fraction, 0–1)")
ax.set_xlabel("")
ax.set_ylim(0, 1.2)
ax.legend(["accuracy (all bridges)", "recall (poor decks)"], loc="upper right", ncol=2)
ax.set_title("Accuracy ranks models one way; recall on poor decks ranks them another")
plt.tight_layout(); plt.show()

**✍️ Question:** Rank the four models by accuracy, then by recall on poor decks. Which ranking matters for the bridge engineer, and why?

*Your answer:* 

## Part 6 — Confusion matrix and the decision threshold (10 min) · 🧩 PROJECT TEMPLATE · Step 6
A **confusion matrix** counts the four possible outcomes. Rows are the truth, columns are the prediction.

In [ ]:
ConfusionMatrixDisplay.from_predictions(y_test, forest.predict(X_test),
                                        display_labels=["not poor", "poor"], cmap="Blues", colorbar=False)
plt.title("Random forest (balanced), test set, threshold 0.5")
plt.xlabel("Predicted deck condition"); plt.ylabel("Actual deck condition")
plt.show()

**✍️ Question:** In bridge terms, what does the bottom-left cell mean, and what does the top-right cell mean? Which kind of error is more serious for public safety?

*Your answer:* 

A classifier actually outputs a **probability** (`predict_proba`). By default it says "poor" when the
probability is ≥ 0.5. That **threshold is a design choice**, like a safety factor: lower it and you catch more
poor decks but flag more bridges.

In [ ]:
proba = forest.predict_proba(X_test)[:, 1]     # probability of "poor" for each test bridge
thresholds = np.arange(0.05, 0.75, 0.05)
recalls = [recall_score(y_test, (proba >= t).astype(int)) for t in thresholds]
precisions = [precision_score(y_test, (proba >= t).astype(int), zero_division=0) for t in thresholds]

plt.figure(figsize=(6, 3.8))
plt.plot(thresholds, recalls, "o-", color="#AB0520", label="recall (poor decks caught)")
plt.plot(thresholds, precisions, "s-", color="#0C234B", label="precision (flags that are right)")
plt.xlabel("Decision threshold on predicted probability")
plt.ylabel("Score (fraction, 0–1)")
plt.title("Lower threshold: catch more poor decks, raise more false alarms")
plt.legend(); plt.grid(alpha=0.3); plt.show()

> 💡 **Hint:** Same pattern as in the plotting cell: `(proba >= 0.2).astype(int)`.

In [ ]:
# TODO: predict "poor" when proba >= 0.2 and report the result as "Random forest, threshold 0.2".
pred_02 = None

report("Random forest, threshold 0.2", y_test.values, pred_02)

In [ ]:
# ✅ Self-check: run this cell to test your work above
assert pred_02 is not None and pred_02.sum() > forest.predict(X_test).sum()
print('Correct!')

**✍️ Question:** With threshold 0.2, what fraction of all 1,129 test bridges would be sent for a closer look, and how many of the 12 poor decks would be caught? If inspectors instead picked bridges at random, roughly how many would they need to visit to catch the same share?

*Your answer:* 

## Part 7 — What drives the predictions? (5 min) · 🧩 PROJECT TEMPLATE · Step 7
A random forest reports a **feature importance** for each input: how much the trees' splits on that feature
helped separate poor from not-poor decks in the training data.

In [ ]:
imp = pd.Series(forest.feature_importances_, index=X.columns).sort_values(ascending=True).tail(10)
plt.figure(figsize=(6, 4))
plt.barh(imp.index, imp.values, color="#1E5288")
plt.xlabel("Importance (share of total, sums to 1 over all 32 features)")
plt.title("Random forest: top 10 features")
plt.tight_layout(); plt.show()

**✍️ Question:** Deck width comes out as one of the most important features. Does that mean narrow decks *cause* deterioration? Give one other explanation.

*Your answer:* 

## Part 8 — The engineering memo (10 min) · 🧩 PROJECT TEMPLATE · Step 8
A model is only useful if a decision-maker understands what it can and cannot do. Write **exactly five
sentences** to the state bridge program manager:
1. **Question** — what you predicted and why it matters.
2. **Data** — what data, how much, and its biggest limit.
3. **Result** — your best model's recall and precision *with numbers*, compared with the baseline.
4. **Drivers** — what the model relies on most (careful with cause and effect).
5. **Caveats and recommended use** — how (and how not) to use it.

**✍️ Question:** Write your 5-sentence engineering memo here.

*Your answer:* 

---
## ⭐ Optional challenges (for when you finish early — not graded)
**Challenge 1 — How stable is recall?** With only 12 poor decks in the test set, one lucky split could make a
model look good. **k-fold cross-validation** checks this: split the bridges into k parts (here k = 5), train on
k − 1 parts, test on the remaining part, and rotate so every part is tested once.

In [ ]:
from sklearn.model_selection import StratifiedKFold

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
for k, (tr, te) in enumerate(cv.split(X, y), start=1):
    rf_k = RandomForestClassifier(n_estimators=300, min_samples_leaf=5, class_weight="balanced", random_state=42)
    rf_k.fit(X.iloc[tr], y.iloc[tr])
    lr_k = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, class_weight="balanced"))
    lr_k.fit(X.iloc[tr], y.iloc[tr])
    p_k = rf_k.predict_proba(X.iloc[te])[:, 1]
    pred_lr = lr_k.predict(X.iloc[te])
    print(f"fold {k}: {y.iloc[te].sum()} poor | forest recall {recall_score(y.iloc[te], (p_k >= 0.5).astype(int)):.0%} "
          f"at 0.5 ({(p_k >= 0.5).sum()} flagged), {recall_score(y.iloc[te], (p_k >= 0.2).astype(int)):.0%} at 0.2 "
          f"({(p_k >= 0.2).sum()} flagged) | balanced log. regression {recall_score(y.iloc[te], pred_lr):.0%} "
          f"({pred_lr.sum()} flagged)")

**✍️ Question:** (Challenge) How much does recall change from fold to fold? Is the ranking "forest vs. logistic regression" from Part 5 stable — and is it even a fair comparison?

*Your answer:* 

**Challenge 2 — Does the model beat a simple rule?** An inspector might just say "look at the oldest
bridges first." Flag the **134 oldest** test bridges (the same number the forest flagged at threshold 0.2) and
count how many of the 12 poor decks that rule catches.

> 💡 **Hint:** `X_test["age_years"].rank(method="first", ascending=False) <= 134` is True for the 134 oldest. Then `y_test[oldest_flag].sum()` counts the poor decks among them.

In [ ]:
# TODO (optional): flag the 134 oldest bridges in X_test and count the poor decks caught.
oldest_flag = None     # True for the 134 largest age_years values in X_test
print("The oldest-134 rule catches", None, "of", y_test.sum(), "poor decks")

**✍️ Question:** (Challenge) Compare the oldest-134 rule with the forest at threshold 0.2 (also 134 flags). Is the model worth its complexity here?

*Your answer:* 

---
## 🎟️ Exit ticket
Three short questions you answer **alone, without AI (AI-Free)**, during the last 10 minutes of lab — even if you have not finished every part above. Use your own words, 1–3 sentences each. (If you must leave early, answer them on your own before you submit.)

**1.** A model scores 98.9% accuracy on the bridge test set. Why is that number not enough to call it a good model?

*Your answer:* 

**2.** What does `class_weight="balanced"` change, and why did it help here?

*Your answer:* 

**3.** Name one caveat that should appear in any memo about this bridge model.

*Your answer:* 

---
## 🤖 AI-use acknowledgment
Did you use an AI tool (ChatGPT, Gemini, Copilot, Claude, Colab's AI features, etc.) while working on this notebook? (The small model that this notebook's own code runs does not count.)
- If **no**, write: *"I did not use AI tools for this assignment."*
- If **yes**, write: *"I acknowledge the use of [tool and version] to [specific use]. Prompts I used included [paste or summarize]. I checked the output by [how I verified it]. I am responsible for this work."*

Same template as the course [AI-use guide](https://github.com/TensorBuildLab/AI-Literacy-CE-ARCE/blob/main/resources/ai-use-guide.md).

*Your statement:* 

## 📤 How to submit
**Due:** Sunday 11:59 pm (Arizona time) of this week, on Brightspace. Late work: see the syllabus late-work table (two grace days per semester, then −10% per day, up to 3 days).

1. **Runtime → Restart session and run all** — make sure everything runs without errors (cells that load a model take a few extra minutes).
2. **File → Download → Download .ipynb**.
3. Upload the `.ipynb` file to the matching assignment on **Brightspace**.